In [91]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from simulations import simulations as sim

# cerebellar ROIs
df1 = sim.y_sim()
df2 = sim.y_sim(seed = 9, region = 'region2')
df3 = sim.y_sim(seed = 99, region = 'region3')
df = pd.concat([df1, df2, df3], axis=0)

# tract ROIs
r1 = sim.y_sim(seed = 1, region = 'tract1')
r2 = sim.y_sim(seed = 2, region = 'tract2')
df = pd.concat([df, r1, r2], axis=0)
df = df*3

In [92]:
def _predict_roi_df(response_df, predict_df):
   # this will give a dataframe for each response_roi-week, you have vlaues of all weeks for each pred_roi
   predictors_df = predict_df.rename(columns = {'regionname': 'pred_region', 'mean': 'pred_mean', 'Week': 'pred_week'})
   roi_df = response_df.merge(predictors_df, on = 'subj_id', how = 'inner')


   return roi_df


response_df = pd.concat([df1, df2, df3], axis = 0)
predict_df = pd.concat([r1, r2], axis = 0)

roi_df = _predict_roi_df(response_df, predict_df)

In [93]:
models = {}
for pred_region in roi_df.pred_region.unique():
    for response_region in roi_df.regionname.unique():
        region_df = roi_df[(roi_df.pred_region == pred_region) & (roi_df.regionname == response_region)]
        model = smf.mixedlm("mean~0+ C(Week)*C(pred_week)*pred_mean", data = region_df, groups = region_df.subj_id).fit(maxiter = 400)
        models[(pred_region, response_region)] = model

To put in the weeks, we can just take the 'weeks' column from results_df and do a re search, where the first T. term is for the response, second is for the predictor.

In [94]:
def _results_df(model):

    fe = model.fe_params
    ci = model.conf_int().loc[fe.index]
    #se_vals = _se_vals(model)

    results = pd.DataFrame({
        'week': model.fe_params.index, # use re string search to get week num later
        'beta': model.fe_params.to_numpy(),
        'bse': model.bse_fe.to_numpy(),
        'converged': model.converged,
        't-val': model.tvalues.loc[fe.index].to_numpy(),
        'p-val': model.pvalues.loc[fe.index].to_numpy(),
        'ci_lower': ci[0].to_numpy(),
        'ci_upper': ci[1].to_numpy(),
        'log_likelihood': model.llf,
        #'se': se_vals
    })

    return results
model6 = models[(pred_region, response_region)]
results6 = _results_df(model6)

In [95]:
test = results6.iloc[23]['week']
pat = f'\d+'
res = re.findall(pat, test)
print(res)

['24', '52']


In [109]:
import re
pat = r'C\(Week\)\[T\.(\d+)\]' # put \ after char for literal
def _week_token(weeks_string, pat):
    week_vals = re.findall(pat, weeks_string)
    return week_vals

results6['resp_week'] = results6['week'].apply(lambda x:_week_token(x,pat))

In [110]:
results6

,week,beta,bse,converged,t-val,p-val,ci_lower,ci_upper,log_likelihood,resp_week
0,C(Week)[0],6.946067e+00,3.665270e-01,True,1.895104e+01,4.329625e-80,6.227687,7.664447,4374.179337,[]
1,C(Week)[4],1.094607e+01,3.665475e-01,True,2.986261e+01,6.021233e-196,10.227647,11.664487,4374.179337,[]
2,C(Week)[12],1.894607e+01,3.664717e-01,True,5.169858e+01,0.000000e+00,18.227796,19.664339,4374.179337,[]
3,C(Week)[24],3.094607e+01,3.665074e-01,True,8.443504e+01,0.000000e+00,30.227726,31.664409,4374.179337,[]
4,C(Week)[52],5.894607e+01,3.665573e-01,True,1.608100e+02,0.000000e+00,58.227628,59.664506,4374.179337,[]
5,C(pred_week)[T.4],3.375454e-01,2.476382e-01,True,1.363059e+00,1.728639e-01,-0.147816,0.822907,4374.179337,[]
6,C(pred_week)[T.12],1.012636e+00,7.429145e-01,True,1.363059e+00,1.728639e-01,-0.443449,2.468722,4374.179337,[]
7,C(pred_week)[T.24],2.025273e+00,1.485829e+00,True,1.363059e+00,1.728639e-01,-0.886899,4.937444,4374.179337,[]
8,C(pred_week)[T.52],4.388091e+00,3.219296e+00,True,1.363059e+00,1.728639e-01,-1.921614,10.697795,4374.179337,[]
9,C(Week)[T.4]:C(pred_week)[T.4],-2.257294e-13,8.554461e-06,True,-2.638733e-08,1.000000e+00,-0.000017,0.000017,4374.179337,[4]


In [ ]:
results6['resp_week'] = results6['week'].apply(_week_token)


SyntaxError: incomplete input (3671139620.py, line 1)

In [ ]:
results6

,week,beta,bse,converged,t-val,p-val,ci_lower,ci_upper,log_likelihood,resp_week
0,C(Week)[0],6.946067e+00,3.665270e-01,True,1.895104e+01,4.329625e-80,6.227687,7.664447,4374.179337,[]
1,C(Week)[4],1.094607e+01,3.665475e-01,True,2.986261e+01,6.021233e-196,10.227647,11.664487,4374.179337,[]
2,C(Week)[12],1.894607e+01,3.664717e-01,True,5.169858e+01,0.000000e+00,18.227796,19.664339,4374.179337,[]
3,C(Week)[24],3.094607e+01,3.665074e-01,True,8.443504e+01,0.000000e+00,30.227726,31.664409,4374.179337,[]
4,C(Week)[52],5.894607e+01,3.665573e-01,True,1.608100e+02,0.000000e+00,58.227628,59.664506,4374.179337,[]
5,C(pred_week)[T.4],3.375454e-01,2.476382e-01,True,1.363059e+00,1.728639e-01,-0.147816,0.822907,4374.179337,[]
6,C(pred_week)[T.12],1.012636e+00,7.429145e-01,True,1.363059e+00,1.728639e-01,-0.443449,2.468722,4374.179337,[]
7,C(pred_week)[T.24],2.025273e+00,1.485829e+00,True,1.363059e+00,1.728639e-01,-0.886899,4.937444,4374.179337,[]
8,C(pred_week)[T.52],4.388091e+00,3.219296e+00,True,1.363059e+00,1.728639e-01,-1.921614,10.697795,4374.179337,[]
9,C(Week)[T.4]:C(pred_week)[T.4],-2.257294e-13,8.554461e-06,True,-2.638733e-08,1.000000e+00,-0.000017,0.000017,4374.179337,[4]


In [ ]:
C\(Week\)\[T\.(\d+)\]

In [ ]:
test = results6.iloc[23]['week']
pat = r'C\(Week\)\[T\.(\d+)\]' # put \ after char for literal
res = re.findall(pat, test)
print(res)

['24']


In [ ]:
results6

,week,beta,bse,converged,t-val,p-val,ci_lower,ci_upper,log_likelihood
0,C(Week)[0],6.946067e+00,3.665270e-01,True,1.895104e+01,4.329625e-80,6.227687,7.664447,4374.179337
1,C(Week)[4],1.094607e+01,3.665475e-01,True,2.986261e+01,6.021233e-196,10.227647,11.664487,4374.179337
2,C(Week)[12],1.894607e+01,3.664717e-01,True,5.169858e+01,0.000000e+00,18.227796,19.664339,4374.179337
3,C(Week)[24],3.094607e+01,3.665074e-01,True,8.443504e+01,0.000000e+00,30.227726,31.664409,4374.179337
4,C(Week)[52],5.894607e+01,3.665573e-01,True,1.608100e+02,0.000000e+00,58.227628,59.664506,4374.179337
5,C(pred_week)[T.4],3.375454e-01,2.476382e-01,True,1.363059e+00,1.728639e-01,-0.147816,0.822907,4374.179337
6,C(pred_week)[T.12],1.012636e+00,7.429145e-01,True,1.363059e+00,1.728639e-01,-0.443449,2.468722,4374.179337
7,C(pred_week)[T.24],2.025273e+00,1.485829e+00,True,1.363059e+00,1.728639e-01,-0.886899,4.937444,4374.179337
8,C(pred_week)[T.52],4.388091e+00,3.219296e+00,True,1.363059e+00,1.728639e-01,-1.921614,10.697795,4374.179337
9,C(Week)[T.4]:C(pred_week)[T.4],-2.257294e-13,8.554461e-06,True,-2.638733e-08,1.000000e+00,-0.000017,0.000017,4374.179337


In [ ]:
results6['pred_week'] = results6['week']

'C(Week)[0]'

In [ ]:
results6.iloc[0]['week']

'C(Week)[0]'

In [ ]:
week_val

'C(Week)[0]'

how do we sum up the betas here?